ex1

In [1]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# Effect size given directly as 0.3
analysis = NormalIndPower()
n = analysis.solve_power(effect_size=0.3, alpha=0.05, power=0.8,
                         ratio=1, alternative='two-sided')
print(f"Required sample size per group: {n:.0f}")

Required sample size per group: 174


ex2

In [2]:
from statsmodels.stats.power import NormalIndPower

analysis = NormalIndPower()
for es in [0.2, 0.4, 0.5]:
    n = analysis.solve_power(effect_size=es, alpha=0.05, power=0.8,
                             ratio=1, alternative='two-sided')
    print(f"Effect size {es}: {n:.0f} per group")

Effect size 0.2: 392 per group
Effect size 0.4: 98 per group
Effect size 0.5: 63 per group


In [ ]:
# Bigger effect size → smaller sample. The relationship is inverse-square (n ∝ 1/effect²), so doubling the effect cuts the sample ~4×. Large effects stand out clearly even in small samples; small effects are buried in noise and need much more data to detect.

ex3

In [3]:
from statsmodels.stats.power import NormalIndPower

analysis = NormalIndPower()
for power in [0.7, 0.8, 0.9]:
    n = analysis.solve_power(effect_size=0.2, alpha=0.05, power=power,
                             ratio=1, alternative='two-sided')
    print(f"Power {power}: {n:.0f} per group")

Power 0.7: 309 per group
Power 0.8: 392 per group
Power 0.9: 525 per group


In [ ]:
# Higher power requires a larger sample (power 0.7 → ~310, 0.8 → ~393, 0.9 → ~526 per group). More power means a lower chance of missing a real effect, and that extra certainty costs more data. It matters because it's a trade-off: too little power risks abandoning a real winner (false negative), while too much costs more traffic and time. The right level is a business call weighing test cost against the cost of a missed opportunity.

ex4

In [4]:
import numpy as np
from scipy.stats import norm

# O'Brien-Fleming boundaries for K equally spaced looks
def obf_boundaries(K, alpha=0.05):
    z_final = norm.ppf(1 - alpha/2)
    looks = np.arange(1, K+1)
    info_frac = looks / K
    z_bounds = z_final / np.sqrt(info_frac)          # O'Brien-Fleming z-boundaries
    p_bounds = 2 * (1 - norm.cdf(z_bounds))          # two-sided p-value thresholds
    return info_frac, z_bounds, p_bounds

frac, z, p = obf_boundaries(K=4, alpha=0.05)
for i in range(4):
    print(f"Week {i+1}: stop if p < {p[i]:.4f}")

Week 1: stop if p < 0.0001
Week 2: stop if p < 0.0056
Week 3: stop if p < 0.0236
Week 4: stop if p < 0.0500


In [ ]:
# Stopping criteria: Use alpha-spending (e.g., O'Brien-Fleming) to split the total 0.05 across the planned weekly looks. Stop early for success only if a look's p-value crosses that look's adjusted boundary; optionally stop for futility; otherwise continue to the final look.
# Implementation: Fix the number of looks and boundaries before starting. Each week, compute the test statistic on accumulated data and compare its p-value to that week's pre-set threshold rather than to 0.05.
# Week 3, p = 0.02: Don't treat it as significant just because it's under 0.05 — with three peeks the threshold is stricter (≈0.019 under O'Brien-Fleming). Since 0.02 is just above the boundary, continue to week 4 and decide at the final planned look rather than stopping now.

ex5

In [ ]:
# Prior setup: Start with a non-informative/neutral prior reflecting 50% belief — e.g. a Beta(1,1) (uniform) prior on the conversion rate, or equal priors on "B better" vs "A better." This says you have no initial lean either way.
# Posterior at 65%: After updating with data, there's a 65% probability the new feature is better. This is a moderate lean in favor but not strong evidence. I'd likely treat it as promising — either roll out cautiously or, better, keep collecting data to push the probability higher (e.g. toward 90–95%) before fully committing.
# If posterior were 55%: That's barely above a coin flip — essentially inconclusive. I would not ship based on it. Keep the test running to gather more data, since the result is too uncertain to act on either way.

ex6


In [ ]:
# Adjust after week 1: Shift more traffic toward Layout C since it's performing best, while keeping some traffic on A and B to keep learning. Don't cut them off entirely — e.g. move to something like 50% C, 25% A, 25% B. Use a principled rule (multi-armed bandit, e.g. Thompson sampling) rather than picking percentages by hand, so allocation reflects the probability each layout is best.
# Continue adapting: Each week, re-update allocations based on cumulative results. Layouts that keep winning get progressively more traffic; weak ones shrink toward zero. Over time traffic converges on the best layout (exploitation) while still occasionally testing others (exploration).
# Challenges & fixes:

# Early noise — a layout can look good by chance in week 1. Fix: don't over-commit early; require enough samples before shifting heavily.
# Time/novelty effects — engagement may shift due to weekday/seasonality, not the layout. Fix: account for time trends; avoid locking in too fast.
# Reduced statistical validity — adaptive allocation complicates classic significance tests. Fix: use Bayesian/bandit methods designed for it.
# Premature convergence — killing an arm too soon may discard a true winner. Fix: keep a minimum exploration floor.